In [ ]:
from os import listdir as ls
from matplotlib import pyplot as plt
import matplotlib.colors as mcolors
import arviz as az

from emu_renewal.inputs import get_world_shp
from emu_renewal.constants import OUTPUTS_PATH, OXCGRT_LOCATION_CMAP

In [ ]:
world = get_world_shp()
world["geometry"] = world.simplify(tolerance=0.1, preserve_topology=True)

job_path = OUTPUTS_PATH / "48930936"
all_countries = [iso3 for iso3 in ls(job_path) if "oxcgrt" in ls(job_path / iso3)]
best_policy = {}
for iso3 in all_countries:
    analysis_path = job_path / iso3 / "oxcgrt"
    idata = az.from_netcdf(analysis_path / "idata_filtered.nc")
    medians = idata.posterior["mob_weights"].median(dim=("chain", "draw"))
    best_policy[iso3] = int(medians.argmax().item())
world["best_policy"] = world["ISO_A3"].map(best_policy).astype("Int64")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 10))
ax.set_xticks([])
ax.set_yticks([])
cmap = mcolors.ListedColormap(OXCGRT_LOCATION_CMAP.values(), name="policy_map")
world.plot(
    ax=ax, 
    column="best_policy", 
    edgecolor="black", 
    cmap=cmap,
)
missing = world[world["best_policy"].isna()]
missing.plot(
    ax=ax,
    facecolor="white",
    edgecolor="black",
    hatch="//",
    alpha=0.12,
)